# Filtering and Sorting Data


In [1]:
import pandas as pd
import numpy as np

## Primer


In [2]:
# Set seed for reproducibility
np.random.seed(999)

# Create a DataFrame
n = 5
data = {"Value": np.random.random(n)}
df = pd.DataFrame(data=data, index=np.arange(1, n + 1, 1))
display(df)

,Value
1,0.803428
2,0.527522
3,0.119111
4,0.639681
5,0.090925


In [3]:
# Create a boolean filter
filt = [True, True, True, False, False]
df.loc[filt, :]

,Value
1,0.803428
2,0.527522
3,0.119111


In [4]:
# Create a boolean filter based on index values
filt = df.index < 4
display(df.loc[filt, :])

# Shorthanded way
display(df[filt])

,Value
1,0.803428
2,0.527522
3,0.119111


,Value
1,0.803428
2,0.527522
3,0.119111


In [5]:
# Create a boolean filter based on column values
filt = df["Value"] > 0.5
df.loc[filt, :]

,Value
1,0.803428
2,0.527522
4,0.639681


In [6]:
# Sort DataFrame by index in descending order
df.sort_index(ascending=False)

,Value
5,0.090925
4,0.639681
3,0.119111
2,0.527522
1,0.803428


In [7]:
# Sort DataFrame by column "Value" in descending order
df.sort_values(by="Value", ascending=False)

,Value
1,0.803428
4,0.639681
2,0.527522
3,0.119111
5,0.090925


## Real Data


In [8]:
# Load the chipotle dataset
chipo = pd.read_csv("./data/chipotle.csv", index_col=0)

In [9]:
# Display first 5 rows of the DataFrame
chipo.head()

,quantity,item_name,choice_description,item_price
order_id,,,,
1,1,Chips and Fresh Tomato Salsa,NaN,$2.39
1,1,Izze,[Clementine],$3.39
1,1,Nantucket Nectar,[Apple],$3.39
1,1,Chips and Tomatillo-Green Chili Salsa,NaN,$2.39
2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",$16.98


In [10]:
# Display DataFrame information
chipo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4622 entries, 1 to 1834
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   quantity            4622 non-null   int64 
 1   item_name           4622 non-null   object
 2   choice_description  3376 non-null   object
 3   item_price          4622 non-null   object
dtypes: int64(1), object(3)
memory usage: 180.5+ KB


### Data pre-processing


In [11]:
# Fill missing values in "choice_description" column with empty strings
chipo["choice_description"] = chipo["choice_description"].fillna("")
chipo.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4622 entries, 1 to 1834
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   quantity            4622 non-null   int64 
 1   item_name           4622 non-null   object
 2   choice_description  4622 non-null   object
 3   item_price          4622 non-null   object
dtypes: int64(1), object(3)
memory usage: 180.5+ KB


In [12]:
# Convert "item_price" column to float after removing the dollar sign
chipo["item_price"] = chipo["item_price"].str.replace("$", "").astype(float)
chipo.head()

,quantity,item_name,choice_description,item_price
order_id,,,,
1,1,Chips and Fresh Tomato Salsa,,2.39
1,1,Izze,[Clementine],3.39
1,1,Nantucket Nectar,[Apple],3.39
1,1,Chips and Tomatillo-Green Chili Salsa,,2.39
2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98


In [13]:
# Create a new column "unit_price" as item_price divided by quantity
chipo["unit_price"] = chipo["item_price"] / chipo["quantity"]
chipo.head()

,quantity,item_name,choice_description,item_price,unit_price
order_id,,,,,
1,1,Chips and Fresh Tomato Salsa,,2.39,2.39
1,1,Izze,[Clementine],3.39,3.39
1,1,Nantucket Nectar,[Apple],3.39,3.39
1,1,Chips and Tomatillo-Green Chili Salsa,,2.39,2.39
2,2,Chicken Bowl,"[Tomatillo-Red Chili Salsa (Hot), [Black Beans...",16.98,8.49


### Show unit price for each item in the dataset

Note that unit prices of the same item may vary. You need to show all unit prices.


In [14]:
# Remove duplicate rows based on "item_name" and "unit_price", keeping the first occurrence
chipo_dd = chipo.drop_duplicates(subset=["item_name", "unit_price"], keep="first")

# Sort the resulting DataFrame by "item_name" (ascending) and "unit_price" (descending)
chipo_dd = chipo_dd.sort_values(by=["item_name", "unit_price"], ascending=[True, False])

# Display the first 10 rows of the final DataFrame
chipo_dd.head(10)

,quantity,item_name,choice_description,item_price,unit_price
order_id,,,,,
129,1,6 Pack Soft Drink,[Sprite],6.49,6.49
19,1,Barbacoa Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",11.75,11.75
1793,1,Barbacoa Bowl,[Guacamole],11.49,11.49
202,1,Barbacoa Bowl,"[[Tomatillo-Green Chili Salsa (Medium), Roaste...",11.48,11.48
42,1,Barbacoa Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",9.25,9.25
51,1,Barbacoa Bowl,"[[Tomatillo-Red Chili Salsa (Hot), Tomatillo-G...",8.99,8.99
1278,1,Barbacoa Bowl,"[Fresh Tomato (Mild), [Lettuce, Black Beans, R...",8.69,8.69
57,1,Barbacoa Burrito,"[Roasted Chili Corn Salsa, [Rice, Pinto Beans,...",11.75,11.75
1313,1,Barbacoa Burrito,"[Fresh Tomato Salsa (Mild), [Black Beans, Rice...",11.48,11.48


### Show item name with unit price > 10.

Name should be unique.


In [15]:
# Create a boolean filter for unit_price > 10
filt = chipo_dd["unit_price"] > 10

# Display item_name of items with unit_price > 10, without duplicates
chipo_dd.loc[filt, ["item_name"]].drop_duplicates(subset=["item_name"], keep="first")

,item_name
order_id,
19,Barbacoa Bowl
57,Barbacoa Burrito
75,Barbacoa Crispy Tacos
501,Barbacoa Salad Bowl
406,Barbacoa Soft Tacos
43,Carnitas Bowl
41,Carnitas Burrito
413,Carnitas Crispy Tacos
468,Carnitas Salad Bowl


### What is the 10 most popular item? (By order count)


In [16]:
# Display the 10 most ordered items
# Note that chipo is used here, not chipo_dd becuase we want total orders, not unique items
chipo["item_name"].value_counts().head(10)

item_name
Chicken Bowl                    726
Chicken Burrito                 553
Chips and Guacamole             479
Steak Burrito                   368
Canned Soft Drink               301
Chips                           211
Steak Bowl                      211
Bottled Water                   162
Chicken Soft Tacos              115
Chips and Fresh Tomato Salsa    110
Name: count, dtype: int64

### What is the stats of unit price of the most popular item? (mean, max, min, ....)


In [17]:
# Create a boolean filter for item_name == "Chicken Bowl"
filt = chipo_dd["item_name"] == "Chicken Bowl"

# Display descriptive statistics for unit_price of "Chicken Bowl"
chipo_dd.loc[filt, ["unit_price"]].describe()

,unit_price
count,8.000000
mean,9.715000
std,1.338368
min,8.190000
25%,8.497500
50%,9.665000
75%,10.980000
max,11.250000


### What is the 10 most expensive items ordered? (by unit price)


In [18]:
# Display the 10 most expensive items based on unit_price
chipo_dd.sort_values(by="unit_price", ascending=False).head(10)

,quantity,item_name,choice_description,item_price,unit_price
order_id,,,,,
123,2,Steak Salad Bowl,"[Tomatillo Red Chili Salsa, [Black Beans, Chee...",23.78,11.89
468,1,Carnitas Salad Bowl,"[Fresh Tomato Salsa, [Rice, Black Beans, Chees...",11.89,11.89
501,1,Barbacoa Salad Bowl,"[Fresh Tomato Salsa, [Rice, Fajita Vegetables,...",11.89,11.89
75,1,Barbacoa Crispy Tacos,"[Tomatillo Red Chili Salsa, [Rice, Black Beans...",11.75,11.75
793,1,Carnitas Soft Tacos,"[Fresh Tomato Salsa, [Sour Cream, Guacamole]]",11.75,11.75
413,1,Carnitas Crispy Tacos,"[Tomatillo Green Chili Salsa, [Rice, Black Bea...",11.75,11.75
41,1,Carnitas Burrito,"[Roasted Chili Corn Salsa, [Sour Cream, Guacam...",11.75,11.75
406,1,Barbacoa Soft Tacos,"[Tomatillo Red Chili Salsa, [Black Beans, Chee...",11.75,11.75
19,1,Barbacoa Bowl,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",11.75,11.75


### How many times was a Veggie Salad Bowl ordered?


In [19]:
# Create a boolean filter for item_name == "Veggie Salad Bowl"
filt = chipo["item_name"] == "Veggie Salad Bowl"

# Display the number of orders for "Veggie Salad Bowl"
chipo[filt].shape[0]

18

### How many times did someone order more than one Canned Soda?


In [20]:
# Create a boolean filter for item_name == "Canned Soda" and quantity > 1
filt = (chipo["item_name"] == "Canned Soda") & (chipo["quantity"] > 1)

# Display the number of orders for "Canned Soda" with quantity > 1
chipo[filt].shape[0]

20

# Choose (unique) items with Fresh Tomato Salsa as ingredient


In [21]:
# Create a boolean filter for choice_description containing "Fresh Tomato Salsa"
filt = chipo_dd["choice_description"].str.contains("Fresh Tomato Salsa")

# Display all columns for items with "Fresh Tomato Salsa" in choice_description
chipo_dd[filt]

,quantity,item_name,choice_description,item_price,unit_price
order_id,,,,,
42,1,Barbacoa Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",9.25,9.25
1313,1,Barbacoa Burrito,"[Fresh Tomato Salsa (Mild), [Black Beans, Rice...",11.48,11.48
36,1,Barbacoa Burrito,"[Fresh Tomato Salsa, [Rice, Pinto Beans, Chees...",9.25,9.25
11,1,Barbacoa Burrito,"[[Fresh Tomato Salsa (Mild), Tomatillo-Green C...",8.99,8.99
501,1,Barbacoa Salad Bowl,"[Fresh Tomato Salsa, [Rice, Fajita Vegetables,...",11.89,11.89
26,1,Barbacoa Soft Tacos,"[Fresh Tomato Salsa, [Fajita Vegetables, Black...",9.25,9.25
43,1,Carnitas Bowl,"[Fresh Tomato Salsa, [Fajita Vegetables, Rice,...",11.75,11.75
438,1,Carnitas Burrito,"[Fresh Tomato Salsa (Mild), [Black Beans, Rice...",11.48,11.48
39,1,Carnitas Burrito,"[Fresh Tomato Salsa, [Rice, Pinto Beans, Sour ...",9.25,9.25


In [22]:
# Add a new column "num_ingredients" that counts the number of ingredients in choice_description
chipo_dd["num_ingredients"] = (
    chipo_dd["choice_description"]
    .str.replace("[", "")
    .str.replace("]", "")
    .str.split(",")
    .apply(len)
)

# Display the 10 items with the highest number of ingredients
chipo_dd[["choice_description", "num_ingredients"]].sort_values(
    by="num_ingredients", ascending=False
).head(10)

,choice_description,num_ingredients
order_id,,
501,"[Fresh Tomato Salsa, [Rice, Fajita Vegetables,...",9
178,"[[Fresh Tomato Salsa (Mild), Tomatillo-Green C...",9
19,"[Roasted Chili Corn Salsa, [Fajita Vegetables,...",8
4,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",8
26,"[Tomatillo Red Chili Salsa, [Fajita Vegetables...",8
189,"[[Roasted Chili Corn Salsa (Medium), Fresh Tom...",8
109,"[Roasted Chili Corn Salsa (Medium), [Black Bea...",8
74,"[Roasted Chili Corn Salsa (Medium), [Pinto Bea...",8
12,"[[Tomatillo-Green Chili Salsa (Medium), Tomati...",8
